# v10.4: Geometric Diffusion — Fixing the Plateau

## What Went Wrong in v10.3

v10.3 correctly removed the SSM scan and manifold coordinates, but introduced
several issues that caused a plateau at ~2.42 CE / BPC 3.50 / 29.6% accuracy
(essentially bigram-level prediction):

1. **Learning rate 1e-5** — ~100x too low for a 5.6M param model. Crawled into
   the nearest shallow basin (character frequencies) and stopped.
2. **No positional encoding** — identical tokens at different positions produced
   identical embeddings. The model couldn't learn position-dependent patterns.
3. **Content-independent mixer** — the diffusion kernel applied the same fixed
   exponential decay regardless of token content. No selectivity.
4. **75% input sparsification** — kept only 16/64 dims per subbundle, destroying
   most of the embedding information before any processing.
5. **Lazy sigmoid residual gate** — could (and did) learn gate ≈ 0, making blocks
   pass-through and ignoring the settled representation entirely.
6. **No FFN** — no per-position nonlinear transformation capacity.
7. **Forward-pass noise** — Langevin noise during training made gradient estimates
   noisy, compounding the too-low LR.

### v10.4 Flow

```
token → embed + sinusoidal PE → [pre-norm → content-gated diffuse → route → settle (deterministic) → residual → FFN → residual] × N → decode
```

### What Changed from v10.3

| Component | v10.3 | v10.4 |
|---|---|---|
| Learning rate | 1e-5 fixed | 3e-4 with cosine warmup |
| Positional encoding | None | Sinusoidal (no extra params) |
| Diffusion mixer | Fixed exp-decay conv | + content gate (selective mixing) |
| Input sparsification | 25% kept (16/64) | 50% kept (32/64) |
| Residual connection | Sigmoid gate (lazy) | Pre-norm + direct add |
| FFN | None | SiLU expand-contract (2× width) |
| Langevin noise | Always on | Off during training, on for generation |

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional, Tuple, Dict, List
from dataclasses import dataclass
from collections import Counter
import math
import os
import urllib.request
from tqdm.notebook import tqdm

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

data_dir = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "v8_real_text", "data")
data_path = os.path.join(data_dir, "input.txt")

if not os.path.exists(data_path):
    os.makedirs(data_dir, exist_ok=True)
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    print("Downloading Tiny Shakespeare...")
    urllib.request.urlretrieve(url, data_path)

with open(data_path, "r") as f:
    text = f.read()

def encode(s): return [ord(c) for c in s]
def decode(t): return "".join(chr(x) for x in t if 0 <= x < 256)

data = torch.tensor(encode(text), dtype=torch.long)
split = int(0.9 * len(data))
train_data, val_data = data[:split], data[split:]
print(f"Train: {len(train_data):,} | Val: {len(val_data):,} chars")

Device: cpu
Train: 1,003,854 | Val: 111,540 chars


In [14]:
@dataclass
class ArchitectureConfig:
    fiber_dim: int = 512
    n_subbundles: int = 8
    context_dim: int = 128

    vocab_size: int = 256
    max_seq_len: int = 512

    atoms_per_subbundle: int = 192
    k_active: int = 16

    n_blocks: int = 4
    langevin_steps: int = 6
    langevin_lr: float = 0.1
    beta_init: float = 1.0
    beta_final: float = 10.0

    sparsity_lambda: float = 0.01
    ffn_mult: int = 2

    learning_rate: float = 3e-4
    warmup_steps: int = 500
    dropout: float = 0.1
    batch_size: int = 4
    seq_len: int = 512
    max_steps: int = 10000
    eval_interval: int = 200
    eval_steps: int = 10

    @property
    def subbundle_dim(self):
        assert self.fiber_dim % self.n_subbundles == 0
        return self.fiber_dim // self.n_subbundles

config = ArchitectureConfig()
print(f"Fiber: {config.fiber_dim} = {config.n_subbundles} x {config.subbundle_dim}")
print(f"Context dim: {config.context_dim} (compressed diffusion context for routing)")
print(f"Atoms: {config.atoms_per_subbundle} per subbundle, k={config.k_active} active")
print(f"Sequence: {config.seq_len} chars, batch: {config.batch_size}")
print(f"LR: {config.learning_rate} with {config.warmup_steps}-step cosine warmup")
print(f"FFN expansion: {config.ffn_mult}x ({config.fiber_dim} -> {config.fiber_dim * config.ffn_mult} -> {config.fiber_dim})")

def get_batch(split_data, cfg):
    max_start = len(split_data) - cfg.seq_len - 1
    starts = torch.randint(0, max_start, (cfg.batch_size,))
    return torch.stack([split_data[s:s+cfg.seq_len] for s in starts]).to(device)

Fiber: 512 = 8 x 64
Context dim: 128 (compressed diffusion context for routing)
Atoms: 192 per subbundle, k=16 active
Sequence: 512 chars, batch: 4
LR: 0.0003 with 500-step cosine warmup
FFN expansion: 2x (512 -> 1024 -> 512)


In [15]:
class SparseTokenEmbedding(nn.Module):
    """Token + position -> sparse fiber bundle section."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.fiber_dim)
        self.topk_per_sub = max(1, cfg.subbundle_dim // 2)  # 50% kept (was 25%)

        # Sinusoidal positional encoding (no learned params)
        pe = torch.zeros(cfg.max_seq_len, cfg.fiber_dim)
        pos = torch.arange(cfg.max_seq_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, cfg.fiber_dim, 2).float() * (-math.log(10000.0) / cfg.fiber_dim))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))  # [1, max_seq_len, fiber_dim]

    def forward(self, token_ids):
        B, T = token_ids.shape
        x = self.embedding(token_ids) + self.pe[:, :T, :]

        chunks = x.chunk(self.cfg.n_subbundles, dim=-1)
        sparse_chunks = []
        for c in chunks:
            _, idx = torch.topk(c.abs(), self.topk_per_sub, dim=-1)
            mask = torch.zeros_like(c)
            mask.scatter_(-1, idx, 1.0)
            sparse_chunks.append(c * mask)
        return torch.cat(sparse_chunks, dim=-1)


class CausalDiffusionMixer(nn.Module):
    """Content-gated causal exponential-decay convolution.

    Same FFT-based causal convolution as v10.3, but with a content-dependent
    gate that controls which dimensions pass context through vs stay local.
    """

    def __init__(self, cfg):
        super().__init__()
        self.log_decay = nn.Parameter(torch.linspace(-1.0, 2.0, cfg.fiber_dim))
        self.gate_proj = nn.Linear(cfg.fiber_dim, cfg.fiber_dim)
        self.norm = nn.LayerNorm(cfg.fiber_dim)

    def forward(self, x_seq):
        B, T, D = x_seq.shape
        decay = F.softplus(self.log_decay)
        lags = torch.arange(T, device=x_seq.device).float()
        kernel = torch.exp(-decay.unsqueeze(0) * lags.unsqueeze(-1))
        kernel = kernel / (kernel.sum(0, keepdim=True) + 1e-8)

        pad_len = T
        x_p = F.pad(x_seq, (0, 0, 0, pad_len))
        k_p = F.pad(kernel, (0, 0, 0, pad_len))
        X = torch.fft.fft(x_p, dim=1)
        K = torch.fft.fft(k_p, dim=0).unsqueeze(0)
        mixed = torch.fft.ifft(X * K, dim=1).real[:, :T, :]

        # Content-dependent gating: select which dims use context vs stay local
        gate = torch.sigmoid(self.gate_proj(x_seq))
        return self.norm(gate * mixed + (1 - gate) * x_seq)


class MemoryBank(nn.Module):
    """Sparse dictionary with context-aware routing. Unchanged from v10.3."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        sd, K, A = cfg.subbundle_dim, cfg.n_subbundles, cfg.atoms_per_subbundle
        self.dictionaries = nn.ParameterList(
            [nn.Parameter(torch.randn(A, sd) * 0.02) for _ in range(K)]
        )
        self.context_proj = nn.Sequential(
            nn.Linear(cfg.fiber_dim, cfg.context_dim), nn.SiLU()
        )
        self.routers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(cfg.context_dim + sd, A), nn.SiLU(), nn.Linear(A, A)
            ) for _ in range(K)
        ])

    def forward(self, x_context, x_local):
        cfg = self.cfg
        sd = cfg.subbundle_dim
        B, T, _ = x_context.shape

        ctx = self.context_proj(x_context).reshape(B * T, -1)
        x_flat = x_local.reshape(B * T, -1)
        x_chunks = x_flat.split(sd, dim=-1)

        M_list = []
        for chunk, dic, router in zip(x_chunks, self.dictionaries, self.routers):
            D_n = F.normalize(dic, dim=-1)
            logits = router(torch.cat([ctx, chunk], dim=-1))
            _, idx = torch.topk(logits, cfg.k_active, dim=-1)
            M_list.append(D_n[idx])

        return M_list


class LangevinDescent(nn.Module):
    """Hopfield gradient descent. No noise during training for clean gradients."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg

    def hopfield_gradient(self, x_chunk, M_k, beta):
        sim = torch.bmm(M_k, x_chunk.unsqueeze(-1)).squeeze(-1)
        w = F.softmax(beta * sim, dim=-1)
        return -torch.bmm(w.unsqueeze(1), M_k).squeeze(1)

    def forward(self, x_seq, M_list):
        cfg = self.cfg
        B, T, D = x_seq.shape
        sd = cfg.subbundle_dim
        x = x_seq.clone()
        betas = torch.linspace(cfg.beta_init, cfg.beta_final, cfg.langevin_steps)

        for step in range(cfg.langevin_steps):
            beta = betas[step].item()

            x_flat = x.reshape(B * T, D)
            x_chunks = x_flat.split(sd, dim=-1)
            grad_chunks = [
                self.hopfield_gradient(xc, mk, beta)
                for xc, mk in zip(x_chunks, M_list)
            ]
            grad_E = torch.cat(grad_chunks, dim=-1).reshape(B, T, D)
            x = x - cfg.langevin_lr * grad_E

            # Noise only during inference (sampling diversity), not training
            if not self.training:
                noise = math.sqrt(2.0 * cfg.langevin_lr / beta) * torch.randn_like(x)
                x = x + noise

            if step == cfg.langevin_steps - 1 and cfg.sparsity_lambda > 0:
                threshold = cfg.sparsity_lambda * cfg.langevin_lr
                x = torch.sign(x) * F.relu(x.abs() - threshold)

        return x


class FiberFFN(nn.Module):
    """Per-position nonlinear transformation. Expand-contract with SiLU."""

    def __init__(self, cfg):
        super().__init__()
        hidden = cfg.fiber_dim * cfg.ffn_mult
        self.net = nn.Sequential(
            nn.Linear(cfg.fiber_dim, hidden),
            nn.SiLU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(hidden, cfg.fiber_dim),
            nn.Dropout(cfg.dropout),
        )

    def forward(self, x):
        return self.net(x)


class GeometricBlock(nn.Module):
    """Pre-norm residual: settle path + FFN path.

    x = x + settle(norm1(x))    [geometric settling with context]
    x = x + ffn(norm2(x))       [per-position nonlinear capacity]

    No sigmoid gate — the direct residual forces each block to contribute.
    """

    def __init__(self, cfg):
        super().__init__()
        self.norm1 = nn.LayerNorm(cfg.fiber_dim)
        self.norm2 = nn.LayerNorm(cfg.fiber_dim)
        self.mixer = CausalDiffusionMixer(cfg)
        self.memory = MemoryBank(cfg)
        self.settle = LangevinDescent(cfg)
        self.ffn = FiberFFN(cfg)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x):
        # Geometric path: diffuse -> route -> settle
        h = self.norm1(x)
        context = self.mixer(h)
        M_list = self.memory(context, h)
        settled = self.settle(h, M_list)
        x = x + self.dropout(settled)

        # Per-position nonlinear capacity
        x = x + self.ffn(self.norm2(x))
        return x


class GeometricDiffusionCLM(nn.Module):
    """v10.4: Geometric diffusion with positional encoding, content-gated
    mixer, FFN, and pre-norm residuals."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embedding = SparseTokenEmbedding(cfg)
        self.blocks = nn.ModuleList([GeometricBlock(cfg) for _ in range(cfg.n_blocks)])
        self.final_norm = nn.LayerNorm(cfg.fiber_dim)
        self.decoder = nn.Sequential(
            nn.Linear(cfg.fiber_dim, cfg.fiber_dim), nn.SiLU(),
            nn.Dropout(cfg.dropout), nn.Linear(cfg.fiber_dim, cfg.vocab_size),
        )
        self.register_buffer("block_loss_weights", torch.linspace(0.1, 1.0, cfg.n_blocks))

    def forward(self, token_ids, return_repr=False):
        B, T = token_ids.shape
        x = self.embedding(token_ids)

        intermediate_logits = []
        for block in self.blocks:
            x = block(x)
            intermediate_logits.append(self.decoder(self.final_norm(x))[:, :-1, :])

        info = {
            "mean_sparsity": (x.abs() < 1e-3).float().mean().item(),
            "intermediate_logits": intermediate_logits,
            "block_loss_weights": self.block_loss_weights,
        }
        if return_repr:
            info["final_repr"] = x
        return intermediate_logits[-1], info


model = GeometricDiffusionCLM(config).to(device)
n_params = sum(p.numel() for p in model.parameters())
n_embed = sum(p.numel() for p in model.embedding.parameters())
n_mixer = sum(sum(p.numel() for p in blk.mixer.parameters()) for blk in model.blocks)
n_memory = sum(sum(p.numel() for p in blk.memory.parameters()) for blk in model.blocks)
n_settle = sum(sum(p.numel() for p in blk.settle.parameters()) for blk in model.blocks)
n_ffn = sum(sum(p.numel() for p in blk.ffn.parameters()) for blk in model.blocks)
n_decoder = sum(p.numel() for p in model.decoder.parameters()) + sum(p.numel() for p in model.final_norm.parameters())
n_other = n_params - n_embed - n_mixer - n_memory - n_ffn - n_decoder

print(f"Total parameters: {n_params:,}")
print(f"  Embedding:            {n_embed:,} ({100*n_embed/n_params:.1f}%)")
print(f"  Diffusion mixers (4): {n_mixer:,} ({100*n_mixer/n_params:.1f}%)")
print(f"  Memory banks (4):     {n_memory:,} ({100*n_memory/n_params:.1f}%)")
print(f"  Langevin (4):         {n_settle:,} ({100*n_settle/n_params:.1f}%)")
print(f"  FFN (4):              {n_ffn:,} ({100*n_ffn/n_params:.1f}%)")
print(f"  Decoder + final norm: {n_decoder:,} ({100*n_decoder/n_params:.1f}%)")
print(f"  Block norms:          {n_other:,} ({100*n_other/n_params:.1f}%)")
print(f"\nv10.3 was 5,661,952 | v10.2 was 10,770,432")

Total parameters: 8,818,944
  Embedding:            131,072 (1.5%)
  Diffusion mixers (4): 1,056,768 (12.0%)
  Memory banks (4):     3,027,456 (34.3%)
  Langevin (4):         0 (0.0%)
  FFN (4):              4,200,448 (47.6%)
  Decoder + final norm: 395,008 (4.5%)
  Block norms:          8,192 (0.1%)

v10.3 was 5,661,952 | v10.2 was 10,770,432


In [16]:
@torch.no_grad()
def estimate_loss(model, cfg):
    model.eval()
    results = {}
    for name, sd in [("train", train_data), ("val", val_data)]:
        tot_ce, tot_ok, tot_n, tot_sp = 0., 0, 0, 0.
        bces = [0.] * cfg.n_blocks
        for _ in range(cfg.eval_steps):
            b = get_batch(sd, cfg)
            logits, info = model(b)
            tgt = b[:, 1:]
            ce = F.cross_entropy(logits.reshape(-1, cfg.vocab_size), tgt.reshape(-1))
            tot_ce += ce.item()
            tot_ok += (logits.argmax(-1) == tgt).sum().item()
            tot_n += tgt.numel()
            tot_sp += info["mean_sparsity"]
            for i, bl in enumerate(info["intermediate_logits"]):
                bces[i] += F.cross_entropy(bl.reshape(-1, cfg.vocab_size), tgt.reshape(-1)).item()
        n = cfg.eval_steps
        results[name] = {
            "ce": tot_ce/n, "acc": tot_ok/tot_n, "sparsity": tot_sp/n,
            "block_ces": [c/n for c in bces],
        }
    model.train()
    return results


optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=0.01)

# Cosine decay with linear warmup
def lr_lambda(step):
    if step < config.warmup_steps:
        return step / max(1, config.warmup_steps)
    progress = (step - config.warmup_steps) / max(1, config.max_steps - config.warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

history = {
    "step": [], "train_ce": [], "val_ce": [], "train_acc": [], "val_acc": [],
    "train_sparsity": [], "val_sparsity": [],
    "train_bpc": [], "val_bpc": [],
    "block_val_ces": [[] for _ in range(config.n_blocks)],
    "lr": [],
}

print("Training v10.4 (geometric diffusion — plateau fixes)")
print(f"Steps: {config.max_steps}, Batch: {config.batch_size}, Seq: {config.seq_len}")
print(f"LR: {config.learning_rate} with {config.warmup_steps}-step cosine warmup")
print(f"Fixes: positional encoding, content-gated mixer, FFN, pre-norm residual,")
print(f"       50% input sparsification, no Langevin noise during training")
print(f"Previous results:")
print(f"  v10.3:      BPC 3.50 | Acc 30% |  5.7M params (plateau at step 600)")
print(f"  v10.2:      BPC 3.54 | Acc 29% | 10.8M params")
print(f"  v9:         BPC 2.65 | Acc 45% |  3.8M params")
print("=" * 70)

model.train()
pbar = tqdm(range(config.max_steps), desc="Training", unit="step")

for step in pbar:
    if step % config.eval_interval == 0:
        res = estimate_loss(model, config)
        tr, vl = res["train"], res["val"]
        history["step"].append(step)
        for k in ["ce", "acc", "sparsity"]:
            history[f"train_{k}"].append(tr[k])
            history[f"val_{k}"].append(vl[k])
        history["train_bpc"].append(tr["ce"] / math.log(2))
        history["val_bpc"].append(vl["ce"] / math.log(2))
        history["lr"].append(scheduler.get_last_lr()[0])
        for i in range(config.n_blocks):
            history["block_val_ces"][i].append(vl["block_ces"][i])
        tqdm.write(
            f"Step {step:5d} | Train CE: {tr['ce']:.3f} | Val CE: {vl['ce']:.3f} | "
            f"Val BPC: {vl['ce']/math.log(2):.2f} | Val Acc: {vl['acc']:.1%} | "
            f"Sp: {vl['sparsity']:.1%} | LR: {scheduler.get_last_lr()[0]:.2e}"
        )

    batch = get_batch(train_data, config)
    optimizer.zero_grad()
    logits, info = model(batch)
    targets = batch[:, 1:]

    ce_loss = sum(
        w * F.cross_entropy(bl.reshape(-1, config.vocab_size), targets.reshape(-1))
        for bl, w in zip(info["intermediate_logits"], info["block_loss_weights"])
    ) / info["block_loss_weights"].sum()

    dcl, nd = 0., 0
    for blk in model.blocks:
        for d in blk.memory.dictionaries:
            Dn = F.normalize(d, dim=-1)
            g = Dn @ Dn.T
            dcl += (g - torch.eye(g.size(0), device=g.device)).pow(2).mean()
            nd += 1
    dcl /= max(nd, 1)

    loss = ce_loss + 0.1 * dcl
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    scheduler.step()

    with torch.no_grad():
        batch_acc = (logits.detach().argmax(-1) == targets).float().mean().item()
    pbar.set_postfix(
        ce=f"{ce_loss.item():.3f}",
        acc=f"{batch_acc:.1%}",
        bpc=f"{ce_loss.item()/math.log(2):.2f}",
        lr=f"{scheduler.get_last_lr()[0]:.1e}",
    )

res = estimate_loss(model, config)
tr, vl = res["train"], res["val"]
history["step"].append(config.max_steps)
for k in ["ce", "acc", "sparsity"]:
    history[f"train_{k}"].append(tr[k])
    history[f"val_{k}"].append(vl[k])
history["train_bpc"].append(tr["ce"] / math.log(2))
history["val_bpc"].append(vl["ce"] / math.log(2))
history["lr"].append(scheduler.get_last_lr()[0])
for i in range(config.n_blocks):
    history["block_val_ces"][i].append(vl["block_ces"][i])

print("=" * 70)
print(f"FINAL | Val CE: {vl['ce']:.3f} | Val BPC: {vl['ce']/math.log(2):.2f} | "
      f"Val Acc: {vl['acc']:.1%} | Val PPL: {math.exp(vl['ce']):.1f}")

Training v10.4 (geometric diffusion — plateau fixes)
Steps: 10000, Batch: 4, Seq: 512
LR: 0.0003 with 500-step cosine warmup
Fixes: positional encoding, content-gated mixer, FFN, pre-norm residual,
       50% input sparsification, no Langevin noise during training
Previous results:
  v10.3:      BPC 3.50 | Acc 30% |  5.7M params (plateau at step 600)
  v10.2:      BPC 3.54 | Acc 29% | 10.8M params
  v9:         BPC 2.65 | Acc 45% |  3.8M params


Training:   0%|          | 0/10000 [00:00<?, ?step/s]

Step     0 | Train CE: 5.579 | Val CE: 5.579 | Val BPC: 8.05 | Val Acc: 0.6% | Sp: 0.0% | LR: 0.00e+00
Step   200 | Train CE: 2.646 | Val CE: 2.637 | Val BPC: 3.80 | Val Acc: 25.9% | Sp: 0.0% | LR: 1.20e-04


KeyboardInterrupt: 

In [ ]:
steps = history["step"]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0,0].plot(steps, history["train_ce"], 'b-', label='Train')
axes[0,0].plot(steps, history["val_ce"], 'r-', label='Val')
axes[0,0].set_title('Cross-Entropy'); axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(steps, history["train_bpc"], 'b-', label='Train')
axes[0,1].plot(steps, history["val_bpc"], 'r-', label='Val')
axes[0,1].axhline(y=2.65, color='gray', linestyle='--', alpha=0.7, label='v9 (2.65)')
axes[0,1].axhline(y=3.54, color='orange', linestyle=':', alpha=0.7, label='v10.2 (3.54)')
axes[0,1].axhline(y=3.50, color='green', linestyle='-.', alpha=0.7, label='v10.3 (3.50)')
axes[0,1].set_title('Bits per Character'); axes[0,1].legend(fontsize=8); axes[0,1].grid(True, alpha=0.3)

axes[0,2].plot(steps, history["train_acc"], 'b-', label='Train')
axes[0,2].plot(steps, history["val_acc"], 'r-', label='Val')
axes[0,2].axhline(y=0.45, color='gray', linestyle='--', alpha=0.7, label='v9 (45%)')
axes[0,2].axhline(y=0.29, color='orange', linestyle=':', alpha=0.7, label='v10.2 (29%)')
axes[0,2].axhline(y=0.296, color='green', linestyle='-.', alpha=0.7, label='v10.3 (30%)')
axes[0,2].set_title('Accuracy'); axes[0,2].legend(fontsize=8); axes[0,2].grid(True, alpha=0.3)

# LR schedule
axes[1,0].plot(steps, history["lr"], 'purple', lw=2)
axes[1,0].set_title('Learning Rate (cosine + warmup)'); axes[1,0].grid(True, alpha=0.3)
axes[1,0].set_xlabel('Step'); axes[1,0].set_ylabel('LR')

colors = plt.cm.viridis(np.linspace(0.2, 0.9, config.n_blocks))
for i in range(config.n_blocks):
    axes[1,1].plot(steps, history["block_val_ces"][i], color=colors[i], label=f'Block {i}')
axes[1,1].set_title('Per-Block Val CE'); axes[1,1].legend(fontsize=8); axes[1,1].grid(True, alpha=0.3)

# Learned diffusion decay rates across all blocks
with torch.no_grad():
    for i, blk in enumerate(model.blocks):
        decay = F.softplus(blk.mixer.log_decay).cpu().numpy()
        axes[1,2].hist(decay, bins=30, alpha=0.5, label=f'Block {i}', edgecolor='black', linewidth=0.5)
axes[1,2].set_xlabel('Decay rate'); axes[1,2].set_ylabel('Count')
axes[1,2].set_title('Learned Diffusion Decay Rates\n(geometry of the space)')
axes[1,2].legend(fontsize=8); axes[1,2].grid(True, alpha=0.3)

plt.suptitle('v10.4: Geometric Diffusion (plateau fixes) — Training Diagnostics',
             fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

print(f"\nFinal: Val BPC {history['val_bpc'][-1]:.2f} | Val Acc {history['val_acc'][-1]:.1%}")
print(f"v10.3:      BPC 3.50 | Acc 30% (plateau at step 600)")
print(f"v10.2:      BPC 3.54 | Acc 29% (seq_len=64)")
print(f"v10.2-diff: BPC 3.63 | Acc 27% (seq_len=64)")
print(f"v9:         BPC 2.65 | Acc 45% (seq_len=64)")

In [ ]:
@torch.no_grad()
def generate_text(model, prompt_str, max_new=200, temperature=0.8):
    model.eval()
    cfg = model.cfg
    ids = torch.tensor(encode(prompt_str), dtype=torch.long).unsqueeze(0).to(device)
    for _ in range(max_new):
        ctx = ids[:, -cfg.max_seq_len:] if ids.size(1) >= cfg.max_seq_len else ids
        logits, _ = model(ctx)
        probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
        ids = torch.cat([ids, torch.multinomial(probs, 1)], dim=1)
    return decode(ids[0].tolist())

print("=" * 60)
print("TEXT GENERATION (v10.3, temperature=0.8)")
print("=" * 60)
for p in ["ROMEO:\n", "To be or not to ", "The king ", "O, "]:
    print(f"\n--- Prompt: {repr(p)} ---")
    print(generate_text(model, p))

In [ ]:
@torch.no_grad()
def run_diagnostics(model, cfg):
    """Diagnostics adapted for v10.3: no manifold coords, focus on diffusion geometry."""
    model.eval()

    # Test 1: Context sensitivity — do different contexts produce different atom selections?
    text_a = "The brave knight drew his sword and charged into battle"
    text_b = "The quiet child drew his picture and smiled at mother"
    T = min(cfg.seq_len, len(text_a), len(text_b))
    ids_a = torch.tensor(encode(text_a[:T]), dtype=torch.long).unsqueeze(0).to(device)
    ids_b = torch.tensor(encode(text_b[:T]), dtype=torch.long).unsqueeze(0).to(device)

    x_a = model.embedding(ids_a)
    x_b = model.embedding(ids_b)
    ctx_a = model.blocks[0].mixer(x_a)
    ctx_b = model.blocks[0].mixer(x_b)

    context_div = (ctx_a - ctx_b).norm(dim=-1)[0].cpu().numpy()

    sd = cfg.subbundle_dim
    jaccards = []
    for t in range(T):
        sub_jacs = []
        for k, router in enumerate(model.blocks[0].memory.routers):
            chunk_a = x_a[0, t:t+1, k*sd:(k+1)*sd]
            chunk_b = x_b[0, t:t+1, k*sd:(k+1)*sd]
            proj = model.blocks[0].memory.context_proj
            ca = proj(ctx_a[0, t:t+1])
            cb = proj(ctx_b[0, t:t+1])
            logits_a = router(torch.cat([ca, chunk_a], dim=-1))
            logits_b = router(torch.cat([cb, chunk_b], dim=-1))
            _, idx_a = torch.topk(logits_a, cfg.k_active, dim=-1)
            _, idx_b = torch.topk(logits_b, cfg.k_active, dim=-1)
            set_a = set(idx_a[0].cpu().numpy().tolist())
            set_b = set(idx_b[0].cpu().numpy().tolist())
            jac = len(set_a & set_b) / len(set_a | set_b) if set_a | set_b else 1.0
            sub_jacs.append(jac)
        jaccards.append(np.mean(sub_jacs))

    # Test 2: Diffusion kernel shapes per block
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    axes[0,0].plot(context_div, 'b-', lw=2)
    axes[0,0].set_xlabel('Position'); axes[0,0].set_ylabel('||ctx_a - ctx_b||')
    axes[0,0].set_title('Diffusion Context Divergence\n(same chars, different history)')
    axes[0,0].grid(True, alpha=0.3)

    axes[0,1].plot(jaccards, 'g-', lw=2)
    axes[0,1].set_xlabel('Position'); axes[0,1].set_ylabel('Jaccard')
    axes[0,1].set_title('Atom Selection Overlap\n(1.0 = context-blind, <1.0 = context-sensitive)')
    axes[0,1].axhline(y=1.0, color='r', linestyle='--', alpha=0.5)
    axes[0,1].set_ylim(0, 1.1); axes[0,1].grid(True, alpha=0.3)

    # Kernel visualizations per block
    lags = np.arange(min(100, cfg.seq_len))
    for i, blk in enumerate(model.blocks):
        decay = F.softplus(blk.mixer.log_decay).cpu().numpy()
        ax = axes[0, 2] if i < 1 else axes[1, i - 1]
        for pct in [10, 25, 50, 75, 90]:
            d = np.percentile(decay, pct)
            ax.plot(lags, np.exp(-d * lags), label=f'p{pct} (d={d:.2f})')
        ax.set_xlabel('Lag (chars)'); ax.set_ylabel('Kernel weight')
        ax.set_title(f'Block {i} Diffusion Kernels')
        ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

    # Test 3: Holonomy — does swapping two chars affect downstream?
    text_h = "ROMEO:\nBut, soft! What light through yonder window breaks?"
    T_h = min(cfg.seq_len, len(text_h))
    ids_h = torch.tensor(encode(text_h[:T_h]), dtype=torch.long).unsqueeze(0).to(device)
    holonomies = []
    _, info_orig = model(ids_h, return_repr=True)
    repr_orig = info_orig["final_repr"]
    for pos in range(2, T_h - 2):
        ids_sw = ids_h.clone()
        ids_sw[0, pos], ids_sw[0, pos+1] = ids_sw[0, pos+1].item(), ids_sw[0, pos].item()
        _, info_sw = model(ids_sw, return_repr=True)
        holonomies.append(
            (repr_orig[0, pos+2:] - info_sw["final_repr"][0, pos+2:]).norm(dim=-1).mean().item()
        )

    axes[1,2].plot(range(2, T_h-2), holonomies, 'b-o', ms=3)
    axes[1,2].set_xlabel('Swap Position'); axes[1,2].set_ylabel('Downstream ||delta||')
    axes[1,2].set_title('Holonomy (char swap propagation)')
    axes[1,2].axhline(y=0, color='r', linestyle='--', alpha=0.5)
    axes[1,2].grid(True, alpha=0.3)

    plt.suptitle('v10.3 Diagnostics: Geometric Diffusion', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

    print(f"Context divergence: mean={context_div.mean():.4f}")
    print(f"Atom overlap (Jaccard): mean={np.mean(jaccards):.4f}")
    print(f"Holonomy: mean={np.mean(holonomies):.4f}")

run_diagnostics(model, config)

## Summary

### What Changed (v10.3 → v10.4)

v10.3 plateaued at BPC 3.50 / 30% accuracy after ~600 steps — bigram-level
prediction. Seven issues were identified and fixed:

1. **LR 1e-5 → 3e-4 with cosine warmup** — the optimizer can now actually
   explore the loss landscape instead of getting stuck in the first basin.
2. **Sinusoidal positional encoding** — the model can distinguish positions,
   enabling word-boundary and position-dependent patterns.
3. **Content-gated diffusion mixer** — a learned sigmoid gate on the mixer
   output lets the model selectively use or ignore context per-dimension,
   rather than blindly applying the same exponential smoothing to everything.
4. **50% input sparsification** (was 25%) — preserves more embedding information
   while maintaining the sparse geometric structure.
5. **Pre-norm + direct residual** (was sigmoid gate) — forces each block to
   contribute rather than learning to pass through. The sigmoid gate was "lazy"
   and could collapse to gate ≈ 0, making blocks no-ops.
6. **FiberFFN** — per-position SiLU expand-contract (2× width) provides
   nonlinear transformation capacity that was completely absent.
7. **No Langevin noise during training** — deterministic settling gives clean
   gradients. Noise is only used during generation for sampling diversity.

### Architecture Flow

```
token → embed + sinusoidal PE → [pre-norm → content-gated diffuse → route → settle → +residual → FFN → +residual] × 4 → final_norm → decode
```

### What To Watch

1. **Does the loss keep decreasing past step 600?** The LR fix alone should
   break the plateau. If it still flattens, the content-independent core of
   the mixer (fixed exponential decay) may be the next bottleneck.
2. **Do decay rates differentiate across blocks?** With the FFN providing
   per-position capacity, the mixer can specialize in cross-position mixing
   at different scales.
3. **Per-block CE convergence** — with pre-norm residuals, all blocks should
   contribute (decreasing CE from block 0→3). If early blocks have the same
   CE as later blocks, the depth isn't helping.
4. **Content gate statistics** — if the mixer gates cluster near 0 or 1,
   the model has learned which dimensions need context vs stay local. If
   gates are all ~0.5, the gate isn't being used effectively.

### Architecture Comparison

| | v9 | v10.2 | v10.3 | v10.4 |
|---|---|---|---|---|
| Cross-position | Attn in Langevin | SSM + attn refiner | Diffusion only | Content-gated diffusion |
| Position info | Positional emb | SSM scan | None | Sinusoidal PE |
| Per-position FFN | Yes | No | No | Yes (2× SiLU) |
| Residual | Direct add | Direct add | Sigmoid gate | Pre-norm + add |
| Params | 3.8M | 10.8M | 5.7M | ~8.8M |
| BPC | 2.65 | 3.54 | 3.50 | TBD |